# Aegis AML Training Notebook

This notebook keeps the training pipeline as close to the repository code as possible by calling the existing training modules and scripts instead of reimplementing them.

Pipeline order:
1. Optional feature preparation
2. Train the four lens models that `scripts/train_all_models.py` runs in parallel
3. Train the entity model
4. Score training data with the trained models
5. Build `meta_features.csv`
6. Train the meta learner and calibrator

In [ ]:
from __future__ import annotations

import os
import subprocess
import sys
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

REPO_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
BACKEND_DIR = REPO_ROOT / "backend"
NOTEBOOKS_DIR = REPO_ROOT / "notebooks"
DATA_EXTERNAL = REPO_ROOT / "data" / "external"
DATA_PROCESSED = REPO_ROOT / "data" / "processed"
MODELS_DIR = REPO_ROOT / "models"

if not BACKEND_DIR.exists():
    raise RuntimeError(f"Could not find backend directory from {Path.cwd()}")

os.chdir(REPO_ROOT)
print(f"Repo root: {REPO_ROOT}")
print(f"Backend dir: {BACKEND_DIR}")
print(f"Default processed data dir: {DATA_PROCESSED}")
print(f"Models dir: {MODELS_DIR}")

In [ ]:
# Configuration
DATA_DIR = DATA_PROCESSED
INPUT_DIR = DATA_EXTERNAL
SKIP_FEATURES = False

LENS_MODULES = [
    "app.ml.training.train_behavioral",
    "app.ml.training.train_graph",
    "app.ml.training.train_temporal",
    "app.ml.training.train_offramp",
]

print(f"DATA_DIR={DATA_DIR}")
print(f"INPUT_DIR={INPUT_DIR}")
print(f"SKIP_FEATURES={SKIP_FEATURES}")

In [ ]:
def run(cmd: list[str], cwd: Path = BACKEND_DIR) -> None:
    print("+", " ".join(str(part) for part in cmd))
    subprocess.run(cmd, cwd=cwd, check=True)


def run_module(module: str, *, data_dir: Path = DATA_DIR) -> None:
    run([sys.executable, "-m", module, "--data-dir", str(data_dir.resolve())])


def prepare_features(*, input_dir: Path = INPUT_DIR, data_dir: Path = DATA_DIR) -> None:
    run(
        [
            sys.executable,
            "-m",
            "scripts.prepare_features",
            "--input",
            str(input_dir.resolve()),
            "--output",
            str(data_dir.resolve()),
        ]
    )


def train_parallel_lenses(*, data_dir: Path = DATA_DIR) -> None:
    with ThreadPoolExecutor(max_workers=len(LENS_MODULES)) as pool:
        futures = {pool.submit(run_module, module, data_dir=data_dir): module for module in LENS_MODULES}
        for future in as_completed(futures):
            module = futures[future]
            try:
                future.result()
                print(f"Completed: {module}")
            except Exception:
                print(f"Failed: {module}")
                raise


def train_entity(*, data_dir: Path = DATA_DIR) -> None:
    run_module("app.ml.training.train_entity", data_dir=data_dir)


def score_training_data(*, data_dir: Path = DATA_DIR) -> None:
    run([sys.executable, "-m", "scripts.score_training_data", "--data-dir", str(data_dir.resolve())])


def prepare_meta_features(*, data_dir: Path = DATA_DIR) -> None:
    run([sys.executable, "-m", "scripts.prepare_meta_features", "--data-dir", str(data_dir.resolve())])


def train_meta(*, data_dir: Path = DATA_DIR) -> None:
    run_module("app.ml.training.train_meta", data_dir=data_dir)


def train_all(*, skip_features: bool = SKIP_FEATURES, input_dir: Path = INPUT_DIR, data_dir: Path = DATA_DIR) -> None:
    if not skip_features:
        prepare_features(input_dir=input_dir, data_dir=data_dir)
    train_parallel_lenses(data_dir=data_dir)
    train_entity(data_dir=data_dir)
    score_training_data(data_dir=data_dir)
    prepare_meta_features(data_dir=data_dir)
    train_meta(data_dir=data_dir)

## 1. Optional feature preparation

This cell mirrors the `scripts.prepare_features` step from the repo pipeline.

In [ ]:
if not SKIP_FEATURES:
    prepare_features(input_dir=INPUT_DIR, data_dir=DATA_DIR)
else:
    print("Skipping feature preparation")

## 2. Train the four parallel lens models

This matches `scripts/train_all_models.py`, which trains these four modules concurrently before the entity model.

In [ ]:
train_parallel_lenses(data_dir=DATA_DIR)

## 3. Train the entity model

In [ ]:
train_entity(data_dir=DATA_DIR)

## 4. Score training data with the trained models

This produces `train_features_scored.csv`, which is the expected handoff into meta-feature generation.

In [ ]:
score_training_data(data_dir=DATA_DIR)

## 5. Build meta features

In [ ]:
prepare_meta_features(data_dir=DATA_DIR)

## 6. Train the meta learner

This uses the repository's `app.ml.training.train_meta` module, including the held-out split and Platt calibration.

In [ ]:
train_meta(data_dir=DATA_DIR)

## One-cell full pipeline

Use this if you want the same overall order as `scripts/train_all_models.py` in one go.

In [ ]:
# train_all(skip_features=SKIP_FEATURES, input_dir=INPUT_DIR, data_dir=DATA_DIR)

## Quick artifact check

In [ ]:
expected_artifacts = [
    MODELS_DIR / "behavioral" / "xgboost_behavioral.pkl",
    MODELS_DIR / "behavioral" / "autoencoder_behavioral.pt",
    MODELS_DIR / "graph" / "gat_model.pt",
    MODELS_DIR / "graph" / "node_embeddings.npy",
    MODELS_DIR / "entity" / "entity_classifier.pkl",
    MODELS_DIR / "temporal" / "lstm_model.pt",
    MODELS_DIR / "offramp" / "offramp_classifier.pkl",
    MODELS_DIR / "meta" / "meta_model.pkl",
    MODELS_DIR / "artifacts" / "threshold_config.json",
    MODELS_DIR / "artifacts" / "metrics_report.json",
    DATA_DIR / "train_features_scored.csv",
    DATA_DIR / "meta_features.csv",
]

for artifact in expected_artifacts:
    print(f"{'OK' if artifact.exists() else 'MISSING'}  {artifact}")